# Diabetic Retinopathy Grading: Gemini vs. MedGemma vs. RetiZero

**Comparing three vision-language models on ICDR-graded fundus images (APTOS 2019)**

> ## PILOT RUN -- SYNTHETIC DATA, NOT REAL MODEL OUTPUT
>
> This notebook is running in **PILOT_MODE** (the default in `build_notebook.py`). There
> is no GPU, no Kaggle account, and no Gemini/Hugging Face API access in the environment
> that produced this notebook, so every prediction below is a **seeded, deterministic
> synthetic value** -- not a real API call and not real model inference. The dataset
> itself is also synthetic: a generated, balanced, stratified metadata table, not a real
> Kaggle download.
>
> This exists to prove the scoring/reporting pipeline (sampling, checkpointing, metrics,
> plots) works end to end. It proves nothing about how well Gemini, MedGemma, or RetiZero
> actually grade diabetic retinopathy. See `reports/final_report.md` for the honest
> write-up and `build_notebook.py` (`PILOT_MODE=0`) for the real-run path -- real Kaggle
> download, real Gemini API calls, real MedGemma/RetiZero GPU inference -- which has not
> been executed anywhere in this repository yet.


## 0. Configuration

In [1]:
PILOT_MODE = True            # set by build_notebook.py's PILOT_MODE env var at generation time
N_PER_GRADE = 25                      # images sampled per ICDR grade (0-4). 25 -> 125 total images (statistically defensible; drop to 2 for a 10-image smoke test).
RANDOM_SEED = 42                       # change for a different random draw; keep fixed for reproducibility
GEMINI_MODEL = "gemini-3.5-flash"      # current stable Gemini vision model (gemini-2.5-flash is being retired Oct 2026 -- already seeing early "model not found" errors in the wild, so don't use it)
MEDGEMMA_MODEL_ID = "google/medgemma-4b-it"
RETIZERO_REPO = "https://github.com/LooKing9218/RetiZero.git"
RETIZERO_WEIGHTS_GDRIVE_ID = "14bMmnefO73_NL1Xc4x0A5qFNbuI7GqKM"  # from the RetiZero README
RETIZERO_WEIGHTS_PATH = "/content/RetiZero/checkpoints/retizero_weights.pth"
CONTENT_ROOT = "/content"
RESULTS_DIR = "results"
REPORTS_DIR = "reports"

# Maps the integer ICDR grade (as stored in the dataset's "diagnosis" column) to its
# clinical name, used for plot titles and the printed class distribution below.
GRADE_NAMES = {
    0: "No DR",
    1: "Mild NPDR",
    2: "Moderate NPDR",
    3: "Severe NPDR",
    4: "Proliferative DR",
}

print(f"PILOT_MODE = {PILOT_MODE}")
if PILOT_MODE:
    print("PILOT RUN -- synthetic data and seeded mock predictions only. See markdown above.")


PILOT_MODE = True
PILOT RUN -- synthetic data and seeded mock predictions only. See markdown above.


## 1. Build the synthetic, stratified sample

No Kaggle download in pilot mode. `build_pilot_metadata()` below generates a balanced,
stratified table -- `N_PER_GRADE` images per ICDR grade 0-4 -- deterministically from
`RANDOM_SEED`. This is the exact same function used to (re)generate
`tests/test_metadata_subset.csv`, so the committed fixture and this notebook's sample can
never silently drift apart.


In [2]:
import os
import random
import hashlib
import numpy as np
import pandas as pd

os.makedirs(RESULTS_DIR, exist_ok=True)

def build_pilot_metadata(n_per_grade=N_PER_GRADE, seed=RANDOM_SEED):
    """Deterministic synthetic metadata: n_per_grade images per ICDR grade 0-4 (125 total
    at the default N_PER_GRADE=25), balanced and reproducible from `seed` alone -- no
    Kaggle download, no real images. image_path points at a placeholder path; PILOT_MODE
    never opens it, since predictions come from seeded_prediction(), not real inference.
    """
    rows = []
    for grade in range(5):
        for i in range(n_per_grade):
            id_code = f"pilot_{grade}_{i:03d}"
            rows.append({
                "id_code": id_code,
                "image_path": f"synthetic/{id_code}.png",
                "diagnosis": grade,
                "grade_name": GRADE_NAMES[grade],
            })
    rng = random.Random(seed)
    rng.shuffle(rows)
    return rows

sample_df = pd.DataFrame(build_pilot_metadata())
print(f"Built {len(sample_df)} synthetic rows ({N_PER_GRADE} per grade x 5 grades):")
print("Class distribution (synthetic ground truth):")
print(sample_df["diagnosis"].value_counts().sort_index())
sample_df.head()


Built 125 synthetic rows (25 per grade x 5 grades):
Class distribution (synthetic ground truth):
diagnosis
0    25
1    25
2    25
3    25
4    25
Name: count, dtype: int64


,id_code,image_path,diagnosis,grade_name
0,pilot_0_009,synthetic/pilot_0_009.png,0,No DR
1,pilot_3_021,synthetic/pilot_3_021.png,3,Severe NPDR
2,pilot_3_024,synthetic/pilot_3_024.png,3,Severe NPDR
3,pilot_3_023,synthetic/pilot_3_023.png,3,Severe NPDR
4,pilot_2_020,synthetic/pilot_2_020.png,2,Moderate NPDR


## 2. Visualize the sample's class distribution

In [3]:
import matplotlib.pyplot as plt

# No real images to lay out in a grid (PILOT_MODE has none) -- a class-distribution bar
# chart is the pilot-mode equivalent: it shows the sample is genuinely balanced, which is
# the thing that mattered about the sample grid in the real-run notebook.
counts = sample_df["diagnosis"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([GRADE_NAMES[g] for g in counts.index], counts.values, color="#4C72B0")
ax.set_ylabel("Count")
ax.set_title(f"PILOT RUN -- synthetic sample class distribution (N={len(sample_df)})")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/pilot_grade_distribution.png", dpi=150)
plt.show()


## 3. Seeded mock prediction function (used for all three "models")

Every model section below calls this same function with a different `model_name`, which
is enough to give each model an independent (but still fully reproducible) set of
synthetic predictions -- no network calls, no GPU, no API keys.


In [4]:
def seeded_prediction(image_id, model_name, num_classes=5):
    """Deterministic seeded synthetic prediction, uniform over 0..num_classes-1.

    Same (image_id, model_name) always yields the same output, on any machine, any
    process, any Python invocation -- unlike Python's built-in hash(), which is salted
    per-process by default (PYTHONHASHSEED) and therefore NOT reproducible run to run.
    hashlib.sha256 has no such salt, which is exactly why it is used here instead.

    This is a uniform random draw, not a simulated model -- it makes no attempt to look
    like a plausible real model's accuracy or to fake a difference between models. Any
    accuracy/kappa above chance in the pilot results is sampling noise, not a claim about
    real model quality.
    """
    digest = hashlib.sha256(f"{image_id}:{model_name}".encode("utf-8")).hexdigest()
    rng = random.Random(digest)
    return rng.randrange(num_classes)


## 4. Model 1 -- Gemini


In [5]:
def predict_gemini(image_id):
    return seeded_prediction(image_id, "gemini")


## 5. Model 2 -- MedGemma


In [6]:
def predict_medgemma(image_id):
    return seeded_prediction(image_id, "medgemma")


## 6. Model 3 -- RetiZero

RetiZero is genuinely mechanically different from the other two in the real pipeline
(embedding-similarity argmax over 5 label strings, not a generated digit) -- see the
real-mode section below for that detail. In PILOT_MODE it produces a synthetic label the
same way as the other two, since there is no real embedding model running here to be
mechanically different from.


In [7]:
def predict_retizero(image_id):
    return seeded_prediction(image_id, "retizero")


## 7. Run all three "models" over the sample

Deterministic and fast: no network, no GPU, no rate limits, no checkpointing needed
(one pass regenerates the identical result every time).


In [8]:
print("PILOT RUN -- predictions below are synthetic, seeded mock outputs, not real model inference")

results_df = sample_df.copy()
results_df["ground_truth"] = results_df["diagnosis"].astype(int)
results_df["gemini_pred"] = results_df["id_code"].apply(predict_gemini)
results_df["medgemma_pred"] = results_df["id_code"].apply(predict_medgemma)
results_df["retizero_pred"] = results_df["id_code"].apply(predict_retizero)

predictions_path = f"{RESULTS_DIR}/predictions.csv"
out_cols = ["id_code", "image_path", "ground_truth", "gemini_pred", "medgemma_pred", "retizero_pred"]
results_df[out_cols].to_csv(predictions_path, index=False)
print(f"Wrote {len(results_df)} rows to {predictions_path}")
results_df[out_cols].head()


PILOT RUN -- predictions below are synthetic, seeded mock outputs, not real model inference
Wrote 125 rows to results/predictions.csv


,id_code,image_path,ground_truth,gemini_pred,medgemma_pred,retizero_pred
0,pilot_0_009,synthetic/pilot_0_009.png,0,3,1,4
1,pilot_3_021,synthetic/pilot_3_021.png,3,3,4,4
2,pilot_3_024,synthetic/pilot_3_024.png,3,4,0,0
3,pilot_3_023,synthetic/pilot_3_023.png,3,4,1,2
4,pilot_2_020,synthetic/pilot_2_020.png,2,0,3,1


## 8. Score against ground truth

Diabetic retinopathy grades are **ordinal** (grade 3 is "closer to" grade 4 than to grade 0),
so alongside plain accuracy we report:
- **Quadratic-weighted Cohen's kappa** -- the standard metric in the DR-grading literature
  (this is literally the Kaggle competition's own scoring metric); penalizes distant
  misclassifications more than adjacent ones, and corrects for chance agreement
- **Mean absolute error (MAE)** in grade steps
- **Bootstrap 95% CI on accuracy** -- honest uncertainty bounds for small samples
- **Per-class precision/recall** and a **confusion matrix** per model

**PILOT_MODE:** these numbers describe seeded random noise, not model skill -- see
Section 7 above. Chance-level accuracy for 5 balanced classes is 20%; anything reported
below should be read as "is the scoring math correct", not "is a model good".


In [9]:
import json
import numpy as np
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, confusion_matrix,
    mean_absolute_error, classification_report
)

MODELS = ["gemini_pred", "medgemma_pred", "retizero_pred"]
metrics_summary = {}

def bootstrap_ci(y_true, y_pred, n_boot=1000, seed=RANDOM_SEED):
    # Percentile bootstrap 95% CI for accuracy: resample the predictions (with
    # replacement) n_boot times, compute accuracy each time, and take the 2.5th/97.5th
    # percentiles of that distribution as the interval. This is what makes the CI
    # honest about small-sample uncertainty rather than just reporting a point estimate.
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    accs = [accuracy_score(y_true[idx], y_pred[idx])
            for idx in (rng.integers(0, len(y_true), len(y_true)) for _ in range(n_boot))]
    lo, hi = np.quantile(accs, [0.025, 0.975])
    return round(float(lo), 4), round(float(hi), 4)

for model_col in MODELS:
    model_name = model_col.replace("_pred", "")
    # Drop rows where this model's prediction is missing (failed calls from the run
    # loop above) -- each model is scored only on the images it actually predicted.
    valid = results_df.dropna(subset=[model_col])
    if len(valid) == 0:
        metrics_summary[model_name] = {"error": "no valid predictions"}
        continue

    y_true = valid["ground_truth"].astype(int)
    y_pred = valid[model_col].astype(int)
    acc_ci = bootstrap_ci(y_true, y_pred)

    metrics_summary[model_name] = {
        "n_scored": int(len(valid)),
        "n_total": int(len(results_df)),
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "accuracy_95ci": list(acc_ci),
        # weights="quadratic" is what makes this ordinal-aware: a true grade 4 predicted
        # as grade 0 costs more than a true grade 4 predicted as grade 3.
        "quadratic_weighted_kappa": round(cohen_kappa_score(y_true, y_pred, weights="quadratic"), 4),
        "mean_absolute_error_grades": round(mean_absolute_error(y_true, y_pred), 4),
    }
    print(f"=== {model_name} ===")
    print(classification_report(y_true, y_pred, zero_division=0))
    print(f"accuracy 95% CI: {acc_ci}")
    print()

with open(f"{RESULTS_DIR}/metrics_summary.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

with open(f"{RESULTS_DIR}/metrics.txt", "w") as f:
    for model_col in MODELS:
        model_name = model_col.replace("_pred", "").capitalize()
        m = metrics_summary.get(model_col.replace("_pred", ""), {})
        if "error" in m:
            f.write(f"{model_name}: {m['error']}\n")
        else:
            f.write(
                f"{model_name}: N={m['n_scored']}, Accuracy={m['accuracy']:.3f}, "
                f"QWK={m['quadratic_weighted_kappa']:.3f}, MAE={m['mean_absolute_error_grades']:.3f}, "
                f"Accuracy95CI={m['accuracy_95ci']}\n"
            )
print(f"Wrote {RESULTS_DIR}/metrics_summary.json and {RESULTS_DIR}/metrics.txt")

pd.DataFrame(metrics_summary).T


=== gemini ===
              precision    recall  f1-score   support

           0       0.18      0.12      0.14        25
           1       0.24      0.16      0.19        25
           2       0.33      0.32      0.33        25
           3       0.17      0.24      0.20        25
           4       0.19      0.24      0.21        25

    accuracy                           0.22       125
   macro avg       0.22      0.22      0.21       125
weighted avg       0.22      0.22      0.21       125

accuracy 95% CI: (0.144, 0.288)



=== medgemma ===
              precision    recall  f1-score   support

           0       0.31      0.36      0.33        25
           1       0.17      0.16      0.17        25
           2       0.15      0.16      0.16        25
           3       0.25      0.28      0.26        25
           4       0.26      0.20      0.23        25

    accuracy                           0.23       125
   macro avg       0.23      0.23      0.23       125
weighted avg       0.23      0.23      0.23       125

accuracy 95% CI: (0.16, 0.304)



=== retizero ===
              precision    recall  f1-score   support

           0       0.25      0.20      0.22        25
           1       0.24      0.32      0.27        25
           2       0.20      0.24      0.22        25
           3       0.30      0.28      0.29        25
           4       0.17      0.12      0.14        25

    accuracy                           0.23       125
   macro avg       0.23      0.23      0.23       125
weighted avg       0.23      0.23      0.23       125

accuracy 95% CI: (0.16, 0.312)

Wrote results/metrics_summary.json and results/metrics.txt


,n_scored,n_total,accuracy,accuracy_95ci,quadratic_weighted_kappa,mean_absolute_error_grades
gemini,125,125,0.216,"[0.144, 0.288]",-0.0602,1.632
medgemma,125,125,0.232,"[0.16, 0.304]",0.1785,1.416
retizero,125,125,0.232,"[0.16, 0.312]",-0.0697,1.528


In [10]:
import seaborn as sns

# One confusion matrix per model, side by side, so grading errors (e.g. always confusing
# grade 2 and 3) are visible at a glance rather than buried in the summary metrics.
title_prefix = "PILOT RUN -- synthetic predictions -- "
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, model_col in zip(axes, MODELS):
    model_name = model_col.replace("_pred", "")
    valid = results_df.dropna(subset=[model_col])
    if len(valid) == 0:
        ax.set_title(f"{model_name}: no predictions")
        ax.axis("off")
        continue
    cm = confusion_matrix(valid["ground_truth"].astype(int), valid[model_col].astype(int), labels=[0, 1, 2, 3, 4])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=range(5), yticklabels=range(5))
    ax.set_title(title_prefix + model_name)
    ax.set_xlabel("Predicted grade")
    ax.set_ylabel("True grade")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/pilot_confusion_matrices.png", dpi=150)
plt.show()


## 9. Export a results table for the report

In [11]:
# Same numbers as metrics_summary.json, reshaped into a table (one row per model) and
# printed as Markdown so it can be pasted straight into reports/final_report.md.
summary_table = pd.DataFrame(metrics_summary).T
summary_table.to_csv(f"{RESULTS_DIR}/summary_table.csv")
print(summary_table.to_markdown())


|          |   n_scored |   n_total |   accuracy | accuracy_95ci   |   quadratic_weighted_kappa |   mean_absolute_error_grades |
|:---------|-----------:|----------:|-----------:|:----------------|---------------------------:|-----------------------------:|
| gemini   |        125 |       125 |      0.216 | [0.144, 0.288]  |                    -0.0602 |                        1.632 |
| medgemma |        125 |       125 |      0.232 | [0.16, 0.304]   |                     0.1785 |                        1.416 |
| retizero |        125 |       125 |      0.232 | [0.16, 0.312]   |                    -0.0697 |                        1.528 |


## 10. Notes and limitations (read before writing up results)

- **This was a PILOT RUN.** Every prediction above is a seeded synthetic value from
  `seeded_prediction()`, not real model output. Accuracy/kappa/MAE above measure whether
  the scoring code is correct, not whether Gemini/MedGemma/RetiZero can grade DR.
- **Ground truth here is also synthetic** -- a generated, balanced label set, not real
  APTOS grades. The real dataset's ground truth is one grader's opinion, not an
  infallible reference either (published inter-ophthalmologist ICDR agreement sits at
  kappa 0.40-0.65) -- worth keeping in mind for the eventual real run too.
- **The real-run path exists and is structurally complete** (`build_notebook.py` with
  `PILOT_MODE=0`) but has not been executed anywhere in this repository -- no GPU/Kaggle/
  API credentials in the authoring environment. See `reports/final_report.md` for what a
  real run would need.
